In [ ]:
!pip install ultralytics gradio opencv-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.6 MB/s eta 0:00:00


In [ ]:
import gradio as gr
import cv2
import time
from ultralytics import YOLO

# ---------------- LOAD MODEL ----------------
model = YOLO("yolov8n.pt")

# Emission factors (kg CO2 per second of idling)
EMISSION_MAP = {
    "car": 0.02,
    "bus": 0.08,
    "truck": 0.12,
    "motorcycle": 0.01
}

# ---------------- CORE LOGIC ----------------
def analyze_traffic(video, current_waits, total_saved, is_running, frame_idx):
    if not is_running or video is None:
        yield "System Offline", [], current_waits, total_saved, f"{total_saved:.2f} kg CO2", frame_idx
        return

    # Get video path safely
    video_path = video if isinstance(video, str) else video["name"]
    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 30

    # 🔑 Move to NEW frame for every cycle
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)

    ret, frame = cap.read()
    if not ret:
        frame_idx = 0
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        ret, frame = cap.read()

    lane_data = {
        "North": {"count": 0, "emit_rate": 0},
        "South": {"count": 0, "emit_rate": 0},
        "East":  {"count": 0, "emit_rate": 0},
        "West":  {"count": 0, "emit_rate": 0}
    }

    # ---------------- DETECTION ----------------
    if ret:
        h, w, _ = frame.shape
        results = model(frame, verbose=False)[0]

        for box in results.boxes:
            label = model.names[int(box.cls[0])]
            if label not in EMISSION_MAP:
                continue

            x_center = int((box.xyxy[0][0] + box.xyxy[0][2]) / 2)
            y_center = int((box.xyxy[0][1] + box.xyxy[0][3]) / 2)

            # ✅ REALISTIC 4-WAY INTERSECTION LOGIC
            if y_center < h * 0.5 and abs(x_center - w * 0.5) < w * 0.25:
                lane = "North"
            elif y_center >= h * 0.5 and abs(x_center - w * 0.5) < w * 0.25:
                lane = "South"
            elif x_center < w * 0.5:
                lane = "West"
            else:
                lane = "East"

            lane_data[lane]["count"] += 1
            lane_data[lane]["emit_rate"] += EMISSION_MAP[label]

    cap.release()

    # ---------------- ANALYTICS ----------------
    table_data = []
    priority_scores = {}

    for lane in lane_data:
        idling_risk = lane_data[lane]["emit_rate"] * current_waits[lane]
        score = (idling_risk * 10) + (current_waits[lane] * 0.5)
        priority_scores[lane] = score

        table_data.append([
            lane,
            lane_data[lane]["count"],
            round(idling_risk, 3),
            current_waits[lane],
            round(score, 2)
        ])

    # ---------------- SIGNAL DECISION ----------------
    green_lane = max(priority_scores, key=priority_scores.get)

    duration = min(
        max(15 + lane_data[green_lane]["count"] * 3, 15),
        60
    )

    total_saved += lane_data[green_lane]["emit_rate"] * duration

    new_waits = {
        l: 0 if l == green_lane else current_waits[l] + duration
        for l in current_waits
    }

    # ⏩ Advance video for NEXT cycle
    frame_idx += int(fps * 2)

    # ---------------- COUNTDOWN ----------------
    for remaining in range(duration, -1, -1):
        status = f"🟢 GREEN SIGNAL: {green_lane} | ⏱️ {remaining}s"
        if remaining == 0:
            status = "🛑 SWITCHING SIGNAL – RE-SCANNING..."

        yield (
            status,
            table_data,
            new_waits,
            total_saved,
            f"{total_saved:.2f} kg CO2",
            frame_idx
        )
        time.sleep(1)

# ---------------- UI ----------------
with gr.Blocks(theme=gr.themes.Soft(), title="Smart Traffic Management System",css="""
    body { overflow: hidden !important; }
    .gradio-container { min-height: 100svh !important; }
    """) as demo:
    gr.Markdown("# 🚦 Smart Traffic Management System for Emission Reduction")

    waits_state = gr.State({"North": 30, "South": 30, "East": 30, "West": 30})
    savings_state = gr.State(0.0)
    frame_state = gr.State(0)
    running_flag = gr.State(False)

    with gr.Row():
        with gr.Column(scale=1):
            video_in = gr.Video(label="Traffic Video Input")
            carbon_display = gr.Label(
                label="🌱 TOTAL CO2 EMISSION SAVED",
                value="0.00 kg CO2"
            )
            btn_start = gr.Button("🚀 START SYSTEM", variant="primary")
            btn_stop = gr.Button("🛑 STOP SYSTEM", variant="stop")

        with gr.Column(scale=2):
            status_box = gr.Textbox(
                label="Live Signal Status",
                interactive=False
            )
            stats_table = gr.Dataframe(
                headers=[
                    "Lane",
                    "Vehicles",
                    "Idling Risk (kg CO2)",
                    "Wait (s)",
                    "Priority Score"
                ],
                label="Intersection Analytics"
            )

    btn_start.click(
        lambda: True,
        outputs=running_flag
    ).then(
        analyze_traffic,
        inputs=[
            video_in,
            waits_state,
            savings_state,
            running_flag,
            frame_state
        ],
        outputs=[
            status_box,
            stats_table,
            waits_state,
            savings_state,
            carbon_display,
            frame_state
        ]
    ).then(
        analyze_traffic,
        inputs=[
            video_in,
            waits_state,
            savings_state,
            running_flag,
            frame_state
        ],
        outputs=[
            status_box,
            stats_table,
            waits_state,
            savings_state,
            carbon_display,
            frame_state
        ]
    )

    btn_stop.click(lambda: False, outputs=running_flag)

demo.launch()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/tmp/ipython-input-4231519874.py:127: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Smart Traffic Management System",css="""
/tmp/ipython-input-4231519874.py:127: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Smart Traffic Management System",css="""


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3c377a5339c4766f0d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
